<div style="background: linear-gradient(120deg, #1a3a5c 0%, #2d6a9f 60%, #4a9eda 100%); padding: 28px 36px; border-radius: 14px; display: flex; align-items: center; gap: 28px; box-shadow: 0 4px 18px rgba(0,0,0,0.18);">
    <img src='Figures/iteso.jpg' style="height: 110px; border-radius: 8px; background: white; padding: 6px; flex-shrink: 0; box-shadow: 0 2px 8px rgba(0,0,0,0.2);"/>
    <div style="border-left: 2px solid rgba(255,255,255,0.4); padding-left: 28px;">
        <h1 style="margin: 0 0 8px 0; color: white; font-size: 1.5em; line-height: 1.3;">Reducción de dimensionalidad usando el Análisis Discriminante Lineal</h1>
        <h3 style="margin: 0 0 8px 0; color: white; font-size: 1.15em; line-height: 1.3;">Ingeniería de Características</h3>
        <h3 style="margin: 0; color: rgba(255,255,255,0.8); font-weight: normal; font-size: 1.05em;">Maestría en Ciencia de Datos</h3>
    </div>
</div>

El **Análisis Discriminante Lineal** (*Linear Discriminant Analysis*, LDA) es un método **supervisado**: durante el ajuste utiliza las variables originales $X$ y una etiqueta de clase $y$. Busca combinaciones lineales de las variables que tiendan a mantener cercanas las observaciones de una misma clase y separadas las de clases diferentes.

Si cada observación tiene $p$ variables, LDA puede transformarla en menos columnas, llamadas **componentes discriminantes** o `LD1`, `LD2`, etc. Estas columnas son características nuevas: no son una selección de las variables originales y no son, por sí solas, una predicción de clase.

| | PCA | LDA |
|---|---|---|
| Tipo | No supervisado | Supervisado |
| Utiliza etiquetas al ajustar | No | Sí |
| Objetivo de la proyección | Conservar direcciones de mayor varianza | Buscar direcciones que separen clases respecto a su dispersión interna |
| Máximo de componentes | Depende de las variables y observaciones | $\min(p, K-1)$, con $K$ clases |

Por ejemplo, el dataset de Iris tiene cuatro variables y tres especies: puede generar como máximo **dos** componentes LDA. Un problema binario permite como máximo **uno**, independientemente de cuántas variables originales tenga. El número efectivo puede ser menor si los datos no tienen suficiente rango.

> El algoritmo de clasificación LDA también puede predecir clases, sin embargo aquí nos concentraremos en su uso como **transformador de características**. Para transformar observaciones nuevas solo se necesitan sus variables $X$, no sus etiquetas.

### La idea matemática

Sean $\mu_k$ la media de la clase $k$, $\mu$ la media global y $n_k$ el número de observaciones de esa clase. Calculamos dos matrices a partir de los datos **de entrenamiento**:

- **Dispersión dentro de las clases:** $S_W = \sum_{k=1}^{K}\sum_{x_i\in k}(x_i-\mu_k)(x_i-\mu_k)^T$.
- **Dispersión entre clases:** $S_B = \sum_{k=1}^{K}n_k(\mu_k-\mu)(\mu_k-\mu)^T$.

Para una dirección $w$, el criterio de Fisher compara ambas cantidades:

$$J(w)=\frac{w^T S_B w}{w^T S_W w}.$$

Buscamos direcciones con gran separación entre clases respecto a la dispersión interna. En el caso multiclase se obtienen a partir del problema generalizado $S_B w = \lambda S_W w$ y se ordenan por sus autovalores. Finalmente, la proyección es $Z=XW$, donde las columnas de $W$ son las direcciones elegidas. Aquí $X$ ya representa los datos estandarizados con parámetros de entrenamiento.

**Precauciones:** la transformación es lineal, valores atípicos, clases que no se separan linealmente y matrices de dispersión singulares pueden afectar el resultado. La estandarización suele ser útil cuando las variables tienen escalas muy distintas, aunque no es un requisito matemático universal.

# LDA desde cero con NumPy (Dataset Iris)

Usaremos cuatro medidas de flores y tres especies. En esta sección implementamos LDA con numpy. La separación estratificada conserva aproximadamente la proporción de las especies en entrenamiento y prueba.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris(as_frame=True)
X_iris = iris.data
y_iris = iris.target

print('Forma de X:', X_iris.shape)
print('Clases:', list(iris.target_names))
X_iris.head()

In [ ]:
y_iris.map(dict(enumerate(iris.target_names))).value_counts().rename('observaciones').to_frame()

### Separar y estandarizar

Reservamos el 25 % de las observaciones. Calculamos la media y la desviación estándar **solo con entrenamiento** y reutilizamos esos valores para transformar prueba. También LDA se ajustará exclusivamente con `X_train_iris` y `y_train_iris`.

In [ ]:
X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.25, stratify=y_iris, random_state=42
)

media_train = X_train_iris.mean()
desviacion_train = X_train_iris.std(ddof=0)
if (desviacion_train == 0).any():
    raise ValueError('Una variable tiene desviación estándar cero en entrenamiento.')

X_train_std = ((X_train_iris - media_train) / desviacion_train).to_numpy()
X_test_std = ((X_test_iris - media_train) / desviacion_train).to_numpy()
y_train_array = y_train_iris.to_numpy()

print('Entrenamiento:', X_train_std.shape, '| Prueba:', X_test_std.shape)

In [ ]:
pd.DataFrame({'media_train': media_train, 'desviacion_train': desviacion_train}).round(3)

### Calcular $S_W$ y $S_B$

Recorremos cada especie: acumulamos las diferencias respecto a su propia media en $S_W$ y las diferencias entre la media de la especie y la media global en $S_B$. Ambas matrices tienen tamaño $p \times p$.

In [ ]:
p = X_train_std.shape[1]
clases_iris = np.unique(y_train_array)
media_global = X_train_std.mean(axis=0)
S_W = np.zeros((p, p))
S_B = np.zeros((p, p))

for clase in clases_iris:
    X_clase = X_train_std[y_train_array == clase]
    media_clase = X_clase.mean(axis=0)
    diferencias = X_clase - media_clase
    S_W += diferencias.T @ diferencias

    diferencia_medias = (media_clase - media_global).reshape(-1, 1)
    S_B += len(X_clase) * (diferencia_medias @ diferencia_medias.T)


In [ ]:
print('Dispersión dentro de las clases (S_W):')
pd.DataFrame(S_W, index=X_iris.columns, columns=X_iris.columns).round(2)


In [ ]:
print('Dispersión entre clases (S_B):')
pd.DataFrame(S_B, index=X_iris.columns, columns=X_iris.columns).round(2)


In [ ]:
print('Rango de S_B:', np.linalg.matrix_rank(S_B), '| límite K - 1:', len(clases_iris) - 1)

### Encontrar las direcciones discriminantes

En lugar de invertir directamente $S_W$, usamos su descomposición simétrica $S_W=U D U^T$ y calculamos $S_W^{-1/2}=U D^{-1/2} U^T$. Esto nos permite resolver con `numpy.linalg.eigh` el problema simétrico:

$$S_W^{-1/2} S_B S_W^{-1/2}v=\lambda v, \qquad w=S_W^{-1/2}v.$$

Ordenamos los autovalores de mayor a menor y conservamos hasta $\min(p,K-1)$ direcciones. Esta implementación didáctica requiere que $S_W$ sea definida positiva; si no lo es, se detiene con un mensaje explícito. Más adelante veremos opciones de scikit-learn para datos con problemas de rango.

In [ ]:
valores_sw, vectores_sw = np.linalg.eigh(S_W)
tolerancia = np.finfo(float).eps * p * valores_sw.max()
if np.any(valores_sw <= tolerancia):
    raise ValueError('S_W es singular o casi singular, esta implementación necesita más rango.')

sw_inversa_raiz = (vectores_sw * (1 / np.sqrt(valores_sw))) @ vectores_sw.T
matriz_simetrica = sw_inversa_raiz @ S_B @ sw_inversa_raiz
matriz_simetrica = (matriz_simetrica + matriz_simetrica.T) / 2

valores, vectores = np.linalg.eigh(matriz_simetrica)
orden = np.argsort(valores)[::-1]
n_componentes = min(p, len(clases_iris) - 1)
autovalores_lda = valores[orden[:n_componentes]]
W_manual = sw_inversa_raiz @ vectores[:, orden[:n_componentes]]


In [ ]:
print('Autovalores discriminantes:', np.round(autovalores_lda, 4))
residuo = S_B @ W_manual - (S_W @ W_manual) * autovalores_lda
print('Residuo máximo del problema generalizado:', np.max(np.abs(residuo)))
pd.DataFrame(W_manual, index=X_iris.columns, columns=['LD1', 'LD2']).round(4)

### Crear las nuevas características y visualizar

Multiplicamos cada observación estandarizada por la misma matriz $W$. Así transformamos **cuatro variables en dos** tanto para entrenamiento como para prueba. Las etiquetas se muestran en la tabla y en la gráfica solo para interpretar las clases; no forman parte de la matriz de características.

In [ ]:
Z_train_manual = X_train_std @ W_manual
Z_test_manual = X_test_std @ W_manual
caracteristicas_train = pd.DataFrame(Z_train_manual, index=X_train_iris.index, columns=['LD1', 'LD2'])
tabla_train = caracteristicas_train.copy()
tabla_train['especie'] = y_train_iris.map(dict(enumerate(iris.target_names)))

print('Antes:', X_train_std.shape, '| Después:', Z_train_manual.shape)
print('Prueba transformada:', Z_test_manual.shape)
display(tabla_train.head().round(3))

fig, ax = plt.subplots(figsize=(8, 5))
for clase in clases_iris:
    mascara = y_train_array == clase
    ax.scatter(Z_train_manual[mascara, 0], Z_train_manual[mascara, 1],
               label=iris.target_names[clase], alpha=0.75)
ax.set(xlabel='LD1', ylabel='LD2', title='Iris: LDA calculado desde cero (entrenamiento)')
ax.legend(title='Especie')
fig.tight_layout()
plt.show()

# LDA con scikit-learn

`LinearDiscriminantAnalysis` implementa el algoritmo y ofrece los métodos `fit` y `transform`. Primero ajustamos un `StandardScaler` con entrenamiento y lo usamos para estandarizar entrenamiento y prueba. Después ajustamos LDA con los datos de entrenamiento estandarizados y sus etiquetas; por último, transformamos prueba sin proporcionar sus etiquetas. Cada paso se realiza explícitamente.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler

escalador_iris = StandardScaler()
X_train_iris_std = escalador_iris.fit_transform(X_train_iris)
X_test_iris_std = escalador_iris.transform(X_test_iris)

lda_iris = LinearDiscriminantAnalysis(n_components=2, solver='svd')
lda_iris.fit(X_train_iris_std, y_train_iris)
Z_train_sklearn = lda_iris.transform(X_train_iris_std)
Z_test_sklearn = lda_iris.transform(X_test_iris_std)

print('Características originales:', X_train_iris.shape[1])
print('Características LDA:', Z_train_sklearn.shape[1])
print('Forma de prueba:', Z_test_sklearn.shape)
display(pd.DataFrame(Z_train_sklearn, index=X_train_iris.index, columns=['LD1', 'LD2']).head().round(3))

### Comparar la implementación manual con la librería

Las coordenadas no tienen que coincidir exactamente: la escala y el signo de un eje pueden cambiar sin alterar la separación. Comparamos las **correlaciones** entre componentes y las proporciones de los autovalores discriminantes.

En este ejemplo, una correlación de valor absoluto cercano a 1 indica que dos ejes representan la misma dirección salvo por escala y signo. Con autovalores repetidos, distintas bases del mismo subespacio pueden mezclarse y la comparación eje por eje no siempre es suficiente.

`explained_variance_ratio_` resume la proporción de los autovalores discriminantes retenidos: **no es** la proporción de varianza total de las variables originales, ni una medida de exactitud de clasificación.

In [ ]:
correlaciones = np.corrcoef(Z_train_manual.T, Z_train_sklearn.T)[:2, 2:]
print('Correlación entre los ejes calculados sobre entrenamiento:')
display(pd.DataFrame(correlaciones, index=['Manual LD1', 'Manual LD2'],
                     columns=['sklearn LD1', 'sklearn LD2']).round(4))

proporcion_manual = autovalores_lda / autovalores_lda.sum()
proporcion_sklearn = lda_iris.explained_variance_ratio_
display(pd.DataFrame({'Manual': proporcion_manual, 'scikit-learn': proporcion_sklearn},
                     index=['LD1', 'LD2']).round(4))

### Transformar observaciones nuevas

Usamos el escalador y LDA ya ajustados para generar características de observaciones reservadas. Primero estandarizamos las nuevas medidas con `escalador_iris.transform` y después las proyectamos con `lda_iris.transform`. En una aplicación real serían registros nuevos con las mismas columnas, **no** se pasa `y_test_iris` a `transform` ni se vuelve a ajustar ninguno de los dos objetos.

In [ ]:
nuevas_flores = X_test_iris.iloc[:3]
nuevas_flores_std = escalador_iris.transform(nuevas_flores)
nuevas_caracteristicas = pd.DataFrame(
    lda_iris.transform(nuevas_flores_std), index=nuevas_flores.index, columns=['LD1', 'LD2']
)
display(pd.concat([nuevas_flores, nuevas_caracteristicas], axis=1).round(3))

# Prática de Laboratorio: Ejemplo con Dataset Wine, de 13 variables a 2 variables

El conjunto Wine contiene 13 mediciones químicas y tres clases de vino. Como $\min(13, 3-1)=2$, LDA puede condensar las mediciones en dos características supervisadas. Usaremos nuevamente una separación estratificada y ajustaremos el escalador y LDA, por separado, solo con entrenamiento.

In [ ]:
from sklearn.datasets import load_wine

wine = load_wine(as_frame=True)
X_wine, y_wine = wine.data, wine.target
X_train_wine, X_test_wine, y_train_wine, y_test_wine = train_test_split(
    X_wine, y_wine, test_size=0.25, stratify=y_wine, random_state=42
)

escalador_wine = StandardScaler()
X_train_wine_std = escalador_wine.fit_transform(X_train_wine)
X_test_wine_std = escalador_wine.transform(X_test_wine)

lda_wine = LinearDiscriminantAnalysis(n_components=2, solver='svd')
lda_wine.fit(X_train_wine_std, y_train_wine)
Z_train_wine = lda_wine.transform(X_train_wine_std)
Z_test_wine = lda_wine.transform(X_test_wine_std)

wine_ld_train = pd.DataFrame(Z_train_wine, index=X_train_wine.index, columns=['LD1', 'LD2'])
wine_ld_test = pd.DataFrame(Z_test_wine, index=X_test_wine.index, columns=['LD1', 'LD2'])
print('Entrenamiento:', X_train_wine.shape, '→', wine_ld_train.shape)
print('Prueba:', X_test_wine.shape, '→', wine_ld_test.shape)
display(wine_ld_train.head().round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for clase in np.unique(y_train_wine):
    mascara = y_train_wine.to_numpy() == clase
    ax.scatter(Z_train_wine[mascara, 0], Z_train_wine[mascara, 1],
               label=wine.target_names[clase], alpha=0.75)
ax.set(xlabel='LD1', ylabel='LD2', title='Wine: características LDA (entrenamiento)')
ax.legend(title='Clase')
fig.tight_layout()
plt.show()

El atributo `scalings_` contiene los coeficientes de las variables **estandarizadas** en cada dirección discriminante. Los signos de los ejes pueden invertirse sin cambiar la proyección en términos de separación. Estas cifras describen una combinación del modelo, no son efectos causales ni equivalen por sí solas a importancia predictiva.

In [ ]:
pesos_wine = pd.DataFrame(
    lda_wine.scalings_[:, :2],
    index=X_wine.columns, columns=['LD1', 'LD2']
)
display(pesos_wine.round(3))

### Utilizar las características en un modelo posterior

¿Conservar 13 variables o reducirlas a 2? Comparamos una regresión logística sobre las variables originales con otra que recibe las características creadas por LDA. El modelo final es la regresión logística; aquí LDA actúa como **paso intermedio de preprocesamiento**.

Usamos validación cruzada estratificada **solo dentro de entrenamiento**. Recorreremos los pliegues explícitamente: en cada uno creamos un escalador nuevo, ajustamos LDA si corresponde y entrenamos una regresión logística. La porción de validación solo se transforma y se evalúa; nunca se utiliza para ajustar esos objetos. No usamos las características calculadas antes de la validación cruzada, porque sus ejes ya habrían visto las etiquetas de validación.

Elegimos el flujo con mayor *balanced accuracy* media, volvemos a ajustar sus pasos con todo el entrenamiento y lo evaluamos una vez en la prueba reservada. Una reducción de dimensiones no garantiza mejorar la métrica; hay que evaluar el compromiso entre desempeño y número de características.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold

usar_lda_por_flujo = {
    '13 variables originales': False,
    '2 características LDA': True,
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
resultados = []
for nombre, usar_lda in usar_lda_por_flujo.items():
    puntajes = []
    for indices_ajuste, indices_validacion in cv.split(X_train_wine, y_train_wine):
        X_ajuste = X_train_wine.iloc[indices_ajuste]
        X_validacion = X_train_wine.iloc[indices_validacion]
        y_ajuste = y_train_wine.iloc[indices_ajuste]
        y_validacion = y_train_wine.iloc[indices_validacion]

        escalador_pliegue = StandardScaler()
        X_ajuste_preparado = escalador_pliegue.fit_transform(X_ajuste)
        X_validacion_preparado = escalador_pliegue.transform(X_validacion)
        if usar_lda:
            lda_pliegue = LinearDiscriminantAnalysis(n_components=2, solver='svd')
            lda_pliegue.fit(X_ajuste_preparado, y_ajuste)
            X_ajuste_preparado = lda_pliegue.transform(X_ajuste_preparado)
            X_validacion_preparado = lda_pliegue.transform(X_validacion_preparado)

        clasificador_pliegue = LogisticRegression(max_iter=1000)
        clasificador_pliegue.fit(X_ajuste_preparado, y_ajuste)
        pred_validacion = clasificador_pliegue.predict(X_validacion_preparado)
        puntajes.append(balanced_accuracy_score(y_validacion, pred_validacion))

    puntajes = np.array(puntajes)
    resultados.append({'flujo': nombre, 'media_cv': puntajes.mean(),
                       'desviacion_cv': puntajes.std(ddof=1)})

tabla_cv = pd.DataFrame(resultados).set_index('flujo')
display(tabla_cv.round(3))
nombre_elegido = tabla_cv['media_cv'].idxmax()

escalador_final = StandardScaler()
X_train_final = escalador_final.fit_transform(X_train_wine)
X_test_final = escalador_final.transform(X_test_wine)
if usar_lda_por_flujo[nombre_elegido]:
    lda_final = LinearDiscriminantAnalysis(n_components=2, solver='svd')
    lda_final.fit(X_train_final, y_train_wine)
    X_train_final = lda_final.transform(X_train_final)
    X_test_final = lda_final.transform(X_test_final)

modelo_elegido = LogisticRegression(max_iter=1000)
modelo_elegido.fit(X_train_final, y_train_wine)
predicciones = modelo_elegido.predict(X_test_final)
print('Elegido mediante validación cruzada:', nombre_elegido)
print(f'Balanced accuracy en prueba: {balanced_accuracy_score(y_test_wine, predicciones):.3f}')